# 3D Gaussian Splatting Training on Colab

This notebook runs the trimmed `gaussian-splatting` training code in a Colab GPU runtime.

Expected input dataset layout is the official Graphdeco COLMAP-style scene:

```text
scene_root/
  images/
  sparse/0/cameras.bin
  sparse/0/images.bin
  sparse/0/points3D.bin
```

Set Colab runtime to GPU before running the notebook.

In [19]:
import os
import sys
import platform
import subprocess
from pathlib import Path

try:
    import torch
except ImportError as exc:
    raise RuntimeError('PyTorch is not available in this Colab runtime.') from exc

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Torch:', torch.__version__)
print('Torch CUDA:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime is required. In Colab: Runtime > Change runtime type > GPU.')

gpu_name = torch.cuda.get_device_name(0)
gpu_capability = torch.cuda.get_device_capability(0)
print('GPU:', gpu_name)
print('Compute capability:', gpu_capability)

Python: 3.12.13 (main, Mar  3 2026, 15:01:35) [MSC v.1944 64 bit (AMD64)]
Platform: Windows-11-10.0.26200-SP0
Torch: 2.12.1+cu130
Torch CUDA: 13.0
CUDA available: True
GPU: NVIDIA GeForce RTX 5080
Compute capability: (12, 0)


## 1. Place the Project Folder

Use one of these options:

- Put the folder in Google Drive and set `GS_ROOT` to that path.
- If a project folder already exists under `/content`, the next cell will try to auto-detect it.
- Otherwise the next cell `git clone`s `GIT_REPO_URL` (at `GIT_REF`) automatically.

In [20]:
# Optional Drive mount. Enable this if your project or datasets live in Drive.
USE_DRIVE = False

try:
    import google.colab  # type: ignore
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

NOTEBOOK_ROOT = Path.cwd().resolve()

if USE_DRIVE:
    if not IS_COLAB:
        raise RuntimeError('Google Drive mount is only available in Colab.')
    from google.colab import drive
    drive.mount('/content/drive')

# Select the framework loaded by this notebook.
# - 'graphdeco_3dgs': original gaussian-splatting-style project.
# - 'osn_gs': the OSN-GS framework in this workspace.
FRAMEWORK_MODE = 'osn_gs'

# Set GS_ROOT directly if you keep the project somewhere specific, e.g.
# GS_ROOT = Path('/content/drive/MyDrive/OSN-GS')
GS_ROOT = None

# If no existing project folder is found under /content (or Drive), the next
# cell clones this repo instead of asking for a zip upload.
GIT_REPO_URL = 'https://github.com/kn99e39/OSN-GS.git'
GIT_REF = 'voxel-surface-regions'  # Branch, tag, or commit to check out.
PROJECT_CLONE_ROOT = Path('/content/project_src') if IS_COLAB else NOTEBOOK_ROOT / '_project_src'

if FRAMEWORK_MODE == 'osn_gs':
    # OSN-GS now provides a top-level train.py wrapper so this notebook can
    # discover and run it the same way it discovers classic 3DGS projects.
    TRAIN_SCRIPT = 'train.py'
    GAUSSIAN_MODEL_IMPORT = 'osn_gs.gaussian.torch_model:TorchGaussianModel'
    GAUSSIAN_STREAM_HOOK = ''

    # OSN-GS can use diff_gaussian_rasterization if it is already installed,
    # but this notebook does not require building CUDA submodules from OSN-GS.
    EXTRA_PYTHONPATHS = []
    DIFF_GAUSSIAN_EXTENSION_PATH = None
    DIFF_GAUSSIAN_IMPORT = None
    KNN_EXTENSION_PATH = None
    KNN_EXTENSION_IMPORT = None
    LOSS_EXTENSION_PATH = None
    LOSS_EXTENSION_IMPORT = None
    FRAMEWORK_IMPORT_CHECKS = [
        'osn_gs',
        'osn_gs.core.torch_trainer',
        'osn_gs.data.torch_scene',
    ]
else:
    # Framework-specific overrides for a Graphdeco-style 3DGS project.
    TRAIN_SCRIPT = 'train.py'
    GAUSSIAN_MODEL_IMPORT = 'scene.gaussian_model:GaussianModel'
    GAUSSIAN_STREAM_HOOK = 'update_learning_rate'

    # Add any non-extension Python package folders your framework needs.
    EXTRA_PYTHONPATHS = []

    # Set these paths yourself. Leave a path as None to skip that extension completely.
    # Paths may be relative to GS_ROOT or absolute Colab paths.
    DIFF_GAUSSIAN_EXTENSION_PATH = 'submodules/diff-gaussian-rasterization'
    DIFF_GAUSSIAN_IMPORT = 'diff_gaussian_rasterization'

    KNN_EXTENSION_PATH = 'submodules/simple-knn'
    KNN_EXTENSION_IMPORT = 'simple_knn._C'

    LOSS_EXTENSION_PATH = None
    LOSS_EXTENSION_IMPORT = None
    FRAMEWORK_IMPORT_CHECKS = [
        'arguments',
        'scene.gaussian_model',
    ]

CUDA_EXTENSIONS = []
if DIFF_GAUSSIAN_EXTENSION_PATH is not None:
    CUDA_EXTENSIONS.append({
        'path': DIFF_GAUSSIAN_EXTENSION_PATH,
        'import': DIFF_GAUSSIAN_IMPORT,
        'build': True,
    })
if KNN_EXTENSION_PATH is not None:
    CUDA_EXTENSIONS.append({
        'path': KNN_EXTENSION_PATH,
        'import': KNN_EXTENSION_IMPORT,
        'build': True,
    })
if LOSS_EXTENSION_PATH is not None:
    CUDA_EXTENSIONS.append({
        'path': LOSS_EXTENSION_PATH,
        'import': LOSS_EXTENSION_IMPORT,
        'build': True,
    })

import shutil
import subprocess


def is_project_root(path):
    path = Path(path)
    return (path / TRAIN_SCRIPT).exists()


def find_project_roots(search_roots, max_depth=5):
    roots = []
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue
        if is_project_root(root):
            roots.append(root)
        for train_py in root.rglob(Path(TRAIN_SCRIPT).name):
            try:
                rel_depth = len(train_py.relative_to(root).parts)
            except ValueError:
                continue
            if rel_depth <= max_depth and train_py.name == Path(TRAIN_SCRIPT).name:
                roots.append(train_py.parent)

    unique = []
    seen = set()
    for path in roots:
        resolved = path.resolve()
        if resolved not in seen:
            unique.append(path)
            seen.add(resolved)
    return unique


def _repo_dir_name(repo_url):
    name = repo_url.rstrip('/').rsplit('/', 1)[-1]
    return name[:-4] if name.endswith('.git') else (name or 'project')


def clone_project_repo(repo_url, ref, clone_dir):
    clone_dir = Path(clone_dir)
    if is_project_root(clone_dir):
        print('Using existing clone:', clone_dir)
        return clone_dir
    if clone_dir.exists():
        shutil.rmtree(clone_dir)
    clone_dir.parent.mkdir(parents=True, exist_ok=True)
    print(f'Cloning {repo_url} (ref: {ref}) into {clone_dir}')
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', ref, repo_url, str(clone_dir)],
        check=True,
    )
    candidates = find_project_roots([clone_dir])
    if not candidates:
        raise FileNotFoundError(f'Could not find {TRAIN_SCRIPT} inside the cloned repo: {repo_url}')
    return candidates[0]


if GS_ROOT is not None:
    GS_ROOT = Path(GS_ROOT)

if GS_ROOT is None or not GS_ROOT.exists():
    if not IS_COLAB:
        GS_ROOT = NOTEBOOK_ROOT
    else:
        candidates = find_project_roots([Path('/content'), Path('/content/drive/MyDrive')])
        # Prefer a folder named OSN-GS when the notebook is in OSN-GS mode.
        if FRAMEWORK_MODE == 'osn_gs':
            candidates = sorted(candidates, key=lambda path: 0 if path.name == 'OSN-GS' else 1)
        if candidates:
            GS_ROOT = candidates[0]
        else:
            GS_ROOT = clone_project_repo(GIT_REPO_URL, GIT_REF, PROJECT_CLONE_ROOT / _repo_dir_name(GIT_REPO_URL))

if GS_ROOT is None or not is_project_root(GS_ROOT):
    raise FileNotFoundError(f'Project root must contain {TRAIN_SCRIPT}. Got: {GS_ROOT}')

print('IS_COLAB:', IS_COLAB)
print('NOTEBOOK_ROOT:', NOTEBOOK_ROOT)
print('FRAMEWORK_MODE:', FRAMEWORK_MODE)
print('GS_ROOT:', GS_ROOT)
print('train script exists:', (GS_ROOT / TRAIN_SCRIPT).exists())
print('submodules exists:', (GS_ROOT / 'submodules').exists())

IS_COLAB: False
NOTEBOOK_ROOT: C:\Projects\OSN-GS
FRAMEWORK_MODE: osn_gs
GS_ROOT: C:\Projects\OSN-GS
train script exists: True
submodules exists: False


## 2. Install Python Packages and CUDA Extensions

In [21]:
# ninja speeds up PyTorch CUDA extension builds.
!apt-get -qq update
!apt-get -qq install -y ninja-build

%pip install -q plyfile tqdm opencv-python ninja gdown websockets

# Install framework-specific Python dependencies if the uploaded project provides them.
PROJECT_REQUIREMENTS = GS_ROOT / 'requirements.txt'
if PROJECT_REQUIREMENTS.exists():
    print('Installing project requirements:', PROJECT_REQUIREMENTS)
    %pip install -q -r {PROJECT_REQUIREMENTS}


'apt-get' is not recognized as an internal or external command,
operable program or batch file.


'apt-get' is not recognized as an internal or external command,
operable program or batch file.


Note: you may need to restart the kernel to use updated packages.
Installing project requirements: C:\Projects\OSN-GS\requirements.txt
Note: you may need to restart the kernel to use updated packages.


In [22]:
import os
import sys
import subprocess
import importlib
from pathlib import Path
import torch

assert GS_ROOT.exists(), f'Missing project folder: {GS_ROOT}'
os.chdir(GS_ROOT)

major, minor = torch.cuda.get_device_capability(0)
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{major}.{minor}'
os.environ.setdefault('MAX_JOBS', '2')


def resolve_project_path(value):
    path = Path(value)
    return path if path.is_absolute() else GS_ROOT / path


def normalize_extension(spec):
    if isinstance(spec, (tuple, list)):
        rel_path, import_name = spec[:2]
        return {'path': rel_path, 'import': import_name, 'build': True, 'enabled': True}
    return {
        'path': spec['path'],
        'import': spec.get('import'),
        'build': spec.get('build', True),
        'enabled': spec.get('enabled', True),
        'optional': spec.get('optional', False),
    }

extensions = [normalize_extension(spec) for spec in CUDA_EXTENSIONS if normalize_extension(spec).get('enabled', True)]
extension_paths = [GS_ROOT]
extension_paths += [resolve_project_path(spec['path']) for spec in extensions]
extension_paths += [resolve_project_path(path) for path in EXTRA_PYTHONPATHS]

for path in reversed(extension_paths):
    sys.path.insert(0, str(path))


def patch_known_extension_issues(ext_path):
    simple_knn_cu = ext_path / 'simple_knn.cu'
    if simple_knn_cu.exists():
        text = simple_knn_cu.read_text()
        if 'FLT_MAX' in text and '#include <cfloat>' not in text:
            text = text.replace('#include "simple_knn.h"\n', '#include "simple_knn.h"\n#include <cfloat>\n', 1)
            simple_knn_cu.write_text(text)
            print('Patched missing <cfloat> include:', simple_knn_cu)

# Build extensions in-place. This is more reliable in Colab than relying on editable installs.
for spec in extensions:
    ext_path = resolve_project_path(spec['path'])
    import_name = spec.get('import')
    should_build = spec.get('build', True)
    if not ext_path.exists():
        if spec.get('optional', False):
            print('Skipping missing optional extension:', ext_path)
            continue
        raise FileNotFoundError(f'Missing extension source: {ext_path}')

    patch_known_extension_issues(ext_path)
    setup_py = ext_path / 'setup.py'
    if should_build:
        if not setup_py.exists():
            raise FileNotFoundError(f'Extension build requested but setup.py is missing: {setup_py}')
        print('')
        print(f'Building {ext_path.relative_to(GS_ROOT)}')
        subprocess.run(
            [sys.executable, 'setup.py', 'build_ext', '--inplace'],
            cwd=ext_path,
            check=True,
        )
    else:
        print('')
        print(f'Skipping build for {ext_path.relative_to(GS_ROOT)}')

    importlib.invalidate_caches()
    print('Built/shared objects:')
    for so_path in sorted(ext_path.rglob('*.so')):
        print(' ', so_path)
    if import_name:
        module = importlib.import_module(import_name)
        print('Imported:', import_name, 'from', getattr(module, '__file__', '<binary>'))

print('')
print('Configured PYTHONPATH entries:')
for path in extension_paths:
    print(' ', path)
print('')
print('CUDA extensions built/imported for this kernel.')


Configured PYTHONPATH entries:
  C:\Projects\OSN-GS

CUDA extensions built/imported for this kernel.


In [23]:
import sys
import importlib
from pathlib import Path


def resolve_project_path(value):
    path = Path(value)
    return path if path.is_absolute() else GS_ROOT / path

extensions = [normalize_extension(spec) for spec in CUDA_EXTENSIONS if normalize_extension(spec).get('enabled', True)]
extension_paths = [GS_ROOT]
extension_paths += [resolve_project_path(spec['path']) for spec in extensions]
extension_paths += [resolve_project_path(path) for path in EXTRA_PYTHONPATHS]

for path in reversed(extension_paths):
    sys.path.insert(0, str(path))

for spec in extensions:
    import_name = spec.get('import')
    if import_name:
        module = importlib.import_module(import_name)
        print('Imported extension:', import_name, 'from', getattr(module, '__file__', '<binary>'))

for import_name in FRAMEWORK_IMPORT_CHECKS:
    module = importlib.import_module(import_name)
    print('Imported framework module:', import_name, 'from', getattr(module, '__file__', '<package>'))

print('Framework imports completed successfully.')

Imported framework module: osn_gs from C:\Projects\OSN-GS\osn_gs\__init__.py
Imported framework module: osn_gs.core.torch_trainer from C:\Projects\OSN-GS\osn_gs\core\torch_trainer.py
Imported framework module: osn_gs.data.torch_scene from C:\Projects\OSN-GS\osn_gs\data\torch_scene.py
Framework imports completed successfully.


## 3. Configure Dataset and Output Paths

Upload or mount a COLMAP-style scene, then set `DATA_ROOT` and `MODEL_ROOT`.

`DATA_ROOT` must point to a folder that directly contains `images/` and `sparse/`.

In [24]:
# Upload, download, or auto-detect a COLMAP-style dataset zip.
# OSN-GS consumes the same DATA_ROOT layout through `train.py -s DATA_ROOT`.
LOCAL_DATASET_ROOT = NOTEBOOK_ROOT / 'DATASET'

# Prefer DATASET_DRIVE_URL for large datasets. Leave it empty to use file upload.
DATASET_DRIVE_URL = 'https://drive.google.com/file/d/1lIEfFofR2RAA2rh3lNs3cOM2tU21COeY/view?usp=share_link'  # Example: 'https://drive.google.com/file/d/FILE_ID/view?usp=sharing'
UPLOAD_DATASET_ZIP = IS_COLAB
DATA_UNZIP_DIR = Path('/content/data') if IS_COLAB else LOCAL_DATASET_ROOT
DOWNLOADED_DATASET_ZIP = Path('/content/dataset_from_drive.zip') if IS_COLAB else LOCAL_DATASET_ROOT / 'dataset_from_drive.zip'
DATA_ROOT = Path('/content/data/scene') if IS_COLAB else LOCAL_DATASET_ROOT
MODEL_ROOT = Path('/content/output/osn_gs_scene' if FRAMEWORK_MODE == 'osn_gs' else '/content/output/scene') if IS_COLAB else NOTEBOOK_ROOT / 'output' / ('osn_gs_scene' if FRAMEWORK_MODE == 'osn_gs' else 'scene')

# Drive example, if you prefer persistent storage:
# DATA_ROOT = Path('/content/drive/MyDrive/datasets/my_scene')
# MODEL_ROOT = Path('/content/drive/MyDrive/3dgs_outputs/my_scene')
# Local example: NOTEBOOK_ROOT / 'DATASET'

import zipfile
import gdown


def has_3dgs_dataset_layout(path):
    path = Path(path)
    return (path / 'images').exists() and (path / 'sparse').exists()


def find_3dgs_datasets(search_roots, max_depth=5):
    found = []
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue
        if has_3dgs_dataset_layout(root):
            found.append(root)
        for images_dir in root.rglob('images'):
            try:
                rel_depth = len(images_dir.relative_to(root).parts)
            except ValueError:
                continue
            if rel_depth > max_depth:
                continue
            candidate = images_dir.parent
            if has_3dgs_dataset_layout(candidate):
                found.append(candidate)

    unique = []
    seen = set()
    for path in found:
        resolved = path.resolve()
        if resolved not in seen:
            unique.append(path)
            seen.add(resolved)
    return unique


def looks_like_project_zip(zip_path):
    try:
        if not zipfile.is_zipfile(zip_path):
            return False
        with zipfile.ZipFile(zip_path) as zf:
            return any(name == 'train.py' or name.endswith('/train.py') for name in zf.namelist())
    except Exception:
        return False


def find_dataset_zips(search_roots, max_depth=3):
    zips = []
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue
        for zip_path in root.rglob('*.zip'):
            try:
                rel_depth = len(zip_path.relative_to(root).parts)
            except ValueError:
                continue
            if rel_depth <= max_depth and not looks_like_project_zip(zip_path):
                zips.append(zip_path)
    return sorted(zips, key=lambda path: path.stat().st_size if path.exists() else 0, reverse=True)


def extract_dataset_zip(dataset_zip):
    dataset_zip = Path(dataset_zip)
    if not dataset_zip.exists():
        raise FileNotFoundError(f'Missing dataset zip: {dataset_zip}')
    if not zipfile.is_zipfile(dataset_zip):
        raise ValueError(f'Not a valid zip file, or upload is incomplete: {dataset_zip}')

    DATA_UNZIP_DIR.mkdir(parents=True, exist_ok=True)
    print('Extracting dataset zip:', dataset_zip)
    with zipfile.ZipFile(dataset_zip) as zf:
        zf.extractall(DATA_UNZIP_DIR)

    candidates = find_3dgs_datasets([DATA_UNZIP_DIR])
    if not candidates:
        raise FileNotFoundError('Could not find a folder containing both images/ and sparse/ after extraction.')
    return candidates[0]


search_roots = [DATA_ROOT, GS_ROOT, DATA_UNZIP_DIR]
if IS_COLAB:
    search_roots += [Path('/content'), Path('/content/drive/MyDrive')]

candidates = find_3dgs_datasets(search_roots)
if candidates:
    if not IS_COLAB:
        candidates = sorted(candidates, key=lambda path: 0 if path.resolve() == LOCAL_DATASET_ROOT.resolve() else 1)
    DATA_ROOT = candidates[0]
elif IS_COLAB and DATASET_DRIVE_URL.strip():
    print('Downloading dataset from Google Drive...')
    gdown.download(DATASET_DRIVE_URL.strip(), str(DOWNLOADED_DATASET_ZIP), quiet=False, fuzzy=True)
    DATA_ROOT = extract_dataset_zip(DOWNLOADED_DATASET_ZIP)
else:
    zip_search_roots = [GS_ROOT]
    if IS_COLAB:
        zip_search_roots += [Path('/content'), Path('/content/drive/MyDrive')]
    zip_candidates = find_dataset_zips(zip_search_roots)
    valid_zip = None
    for zip_path in zip_candidates:
        if zipfile.is_zipfile(zip_path):
            valid_zip = zip_path
            break
        print('Skipping invalid or incomplete zip:', zip_path)

    if valid_zip is not None:
        DATA_ROOT = extract_dataset_zip(valid_zip)

if UPLOAD_DATASET_ZIP and not has_3dgs_dataset_layout(DATA_ROOT):
    from google.colab import files
    print('Upload your dataset zip now. It should contain images/ and sparse/ somewhere inside.')
    uploaded = files.upload()
    zip_names = [name for name in uploaded.keys() if name.lower().endswith('.zip')]
    if not zip_names:
        raise FileNotFoundError('No .zip file was uploaded.')

    uploaded_zip = Path('/content') / zip_names[0]
    DATA_ROOT = extract_dataset_zip(uploaded_zip)

if not has_3dgs_dataset_layout(DATA_ROOT):
    print('Searched for datasets in:')
    for root in search_roots:
        print(' ', root)
    raise FileNotFoundError('No COLMAP-style dataset found. DATA_ROOT must contain images/ and sparse/.')

print('FRAMEWORK_MODE:', FRAMEWORK_MODE)
print('LOCAL_DATASET_ROOT:', LOCAL_DATASET_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('MODEL_ROOT:', MODEL_ROOT)
print('images exists:', (DATA_ROOT / 'images').exists())
print('sparse exists:', (DATA_ROOT / 'sparse').exists())


FRAMEWORK_MODE: osn_gs
LOCAL_DATASET_ROOT: C:\Projects\OSN-GS\DATASET
DATA_ROOT: C:\Projects\OSN-GS\DATASET
MODEL_ROOT: C:\Projects\OSN-GS\output\osn_gs_scene
images exists: True
sparse exists: True


In [25]:
required_paths = [
    DATA_ROOT / 'images',
    DATA_ROOT / 'sparse',
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    print('Expected DATA_ROOT layout:')
    print('  DATA_ROOT/images')
    print('  DATA_ROOT/sparse')
    print('\nFix one of these:')
    print('  1. Put your dataset in NOTEBOOK_ROOT/DATASET.')
    print('  2. Set DATA_ROOT to the actual dataset folder.')
    print('  3. If running in Colab, upload a dataset zip or mount Drive.')
    raise FileNotFoundError('Dataset is not ready. Missing: ' + ', '.join(missing))

MODEL_ROOT.mkdir(parents=True, exist_ok=True)
print('Dataset looks ready.')


Dataset looks ready.


## 4. Check Renderer WebSocket Relay

Run this before training if you want to confirm that Colab can reach the renderer relay. The renderer should be connected to the same WebSocket URL.


In [26]:
import asyncio
import json
import time

import websockets

# Use the same channel path in the browser renderer and Colab.
# Local-only example: ws://127.0.0.1:8765/garden works only when Colab runs on the same machine, which it does not.
# For real Colab use, replace the host with a public address that reaches your relay server.
RENDERER_WS_URL = 'wss://cotton-credible-primer.ngrok-free.dev/garden'

async def check_renderer_connection(url):
    if 'YOUR_PUBLIC_HOST' in url:
        raise ValueError('Set RENDERER_WS_URL to the public relay URL before running this cell.')

    probe = {
        'type': 'status',
        'message': f"Colab connected to renderer relay at {time.strftime('%H:%M:%S')}",
        'source': 'colab',
    }

    print('Connecting:', url)
    async with websockets.connect(url, max_size=None, ping_interval=20) as ws:
        greeting = await asyncio.wait_for(ws.recv(), timeout=5)
        print('Relay greeting:', greeting)
        await ws.send(json.dumps(probe))
        print('Status probe sent. Check the renderer status panel for the Colab message.')
        return True

await check_renderer_connection(RENDERER_WS_URL)


Connecting: wss://cotton-credible-primer.ngrok-free.dev/garden
Relay greeting: {"type":"hello","message":"Relay connected on channel \"garden\".","channel":"garden"}
Status probe sent. Check the renderer status panel for the Colab message.


True

## 5. Train

Start with a small iteration count to verify the environment, then increase it.

In [27]:
import os
import subprocess
import sys
import time
import threading
import queue
import re
from collections import deque
from pathlib import Path

ITERATIONS = 10000
SAVE_ITERATIONS = [1000, 2000, 5000, 10000]
TEST_ITERATIONS = [10000]
TRAIN_OUTPUT_TAIL_LINES = 40
TRAIN_STATUS_INTERVAL_SECONDS = 5.0
PROGRESS_LOG_INTERVAL = 100
TIMING_LOG_INTERVAL = 100

# WebSocket streaming to the browser renderer. Shared by OSN-GS and Graphdeco-style 3DGS.
# Run the renderer relay check cell first so RENDERER_WS_URL is set.
STREAM_TO_RENDERER = True
STREAM_EVERY = 1000
STREAM_ITERATIONS = SAVE_ITERATIONS
STREAM_MAX_GAUSSIANS = 0  # Set to 0 to stream every Gaussian. Large scenes may be slow.
STREAM_NURBS = True  # OSN-GS includes the visible NURBS intermediate when available.
STREAM_CACHE_DIR = MODEL_ROOT / 'stream_cache'  # Cached JSON snapshots for manual bulk streaming after training.
WRITE_OUTPUT_FILES = False  # OSN-GS: skip slow PLY/NURBS/checkpoint writes when streaming.

# OSN-GS-specific knobs used only when FRAMEWORK_MODE == 'osn_gs'.
OSN_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OSN_BASE_CURVE_COUNT = 8
OSN_VISIBLE_SURFACE_RESOLUTION_U = 8
OSN_VISIBLE_SURFACE_RESOLUTION_V = 4
OSN_VISIBLE_SURFACE_RESOLUTION_SCALE = 4.0
OSN_VISIBLE_SURFACE_FIT_DEVICE = OSN_DEVICE
OSN_VISIBLE_SURFACE_FIT_CHUNK_SIZE = 0
OSN_MAX_SURFACE_CONTROL_POINTS = 65536
OSN_USE_VOXEL_SURFACE_REGIONS = True
OSN_VOXEL_GRID_RESOLUTION = 16
OSN_ADAPTIVE_VOXEL_DENSITY = True
OSN_VOXEL_MAX_SUBDIVISION_DEPTH = 1
OSN_VOXEL_DENSITY_QUANTILE = 0.75
OSN_VOXEL_DENSITY_COVARIANCE_WEIGHT_CAP = 10.0
OSN_VOXEL_NORMAL_KNN = 16
OSN_VOXEL_BOUNDARY_ANGLE_DEGREES = 35.0
OSN_VOXEL_MIN_POINTS_PER_REGION = 1
OSN_COVARIANCE_INIT = 'knn'
OSN_COVARIANCE_KNN_CHUNK_SIZE = 0
OSN_COVARIANCE_MIN_SCALE = 1e-4
OSN_COVARIANCE_MAX_SCALE_RATIO = 0.05
OSN_COVARIANCE_SCALE_MULTIPLIER = 1.0
OSN_UNCERTAIN_SAMPLES_U = 16
OSN_UNCERTAIN_SAMPLES_V = 3
# Voxel topology is initialized once. This interval only checks persistent NURBS quality.
OSN_SURFACE_UPDATE_INTERVAL = 1000
# 0 evaluates every NURBS patch; 16 rotates a bounded patch minibatch each iteration.
OSN_SURFACE_LOSS_PATCH_BUDGET = 16
OSN_SURFACE_RESIDUAL_RATIO_THRESHOLD = 0.03
OSN_SURFACE_RESIDUAL_PATIENCE = 3
OSN_SURFACE_LOCAL_MIN_GAUSSIANS = 64
OSN_SURFACE_LOCAL_MIN_COMPONENT = 16
OSN_ENABLE_LOCAL_SURFACE_CORRECTION = True
# Parametric NURBS fitting: 'lsq' = regularized least-squares with foot-point
# parameter correction (recommended); 'idw' = legacy inverse-distance seed only.
OSN_SURFACE_FIT_MODE = 'lsq'
OSN_SURFACE_DEGREE_U = 2
OSN_SURFACE_DEGREE_V = 2
OSN_SURFACE_FIT_SMOOTHNESS = 1e-4
OSN_SURFACE_FIT_TIKHONOV = 1e-4
OSN_SURFACE_FIT_ROUNDS = 2
OSN_SURFACE_PROJECTION_ITERATIONS = 4
OSN_DENSITY_CONTROL_INTERVAL = 500
OSN_DENSIFY_FROM_ITER = 500
OSN_DENSIFY_UNTIL_ITER = 15000
OSN_DENSIFICATION_INTERVAL = 100
OSN_DENSIFY_GRAD_THRESHOLD = 0.0002
OSN_OPACITY_RESET_INTERVAL = 3000
OSN_SCREEN_SIZE_PRUNE_FROM_ITER = 3000
OSN_ADC_MAX_GAUSSIANS = 0
OSN_ADC_PERCENT_DENSE = 0.01
OSN_ADC_PRUNE_OPACITY_THRESHOLD = 0.005
OSN_ADC_SPLIT_SAMPLES = 2
OSN_ADC_MAX_SCREEN_SIZE = 20.0
OSN_ADC_MAX_SCALE_RATIO = 0.1
OSN_DISABLE_CUDA_RASTERIZER = False
OSN_IMAGE_DOWNSCALE = 1
OSN_MAX_IMAGES = 0
OSN_LOW_VRAM = True
OSN_IMAGE_DEVICE = 'auto'
OSN_TRAIN_RESOLUTION_SCALE = 2
OSN_MAX_UNCERTAIN_GAUSSIANS = 128
OSN_RESUME_CHECKPOINT = ''

# Graphdeco-style 3DGS knobs used only when FRAMEWORK_MODE != 'osn_gs'.
DENSIFY_UNTIL_ITER = 15000
DENSIFICATION_INTERVAL = 100
DENSIFY_GRAD_THRESHOLD = 0.0002
TRAIN_EXTRA_ARGS = []
PATCH_LAZY_COLMAP_IMAGES = True

try:
    import psutil
except Exception:
    psutil = None

try:
    from IPython.display import clear_output
except Exception:
    def clear_output(wait=False):
        return None

_ITER_RE = re.compile(r'iteration=(\d+)(?:/(\d+))?')

def _gb(value):
    return value / (1024 ** 3)

def _ram_status(process=None):
    parts = []
    if psutil is not None:
        vm = psutil.virtual_memory()
        parts.append(f'RAM {_gb(vm.used):.1f}/{_gb(vm.total):.1f} GB ({vm.percent:.0f}%)')
        if process is not None:
            try:
                child = psutil.Process(process.pid)
                rss = child.memory_info().rss
                for proc in child.children(recursive=True):
                    try:
                        rss += proc.memory_info().rss
                    except Exception:
                        pass
                parts.append(f'train RSS {_gb(rss):.1f} GB')
            except Exception:
                pass
    return ' | '.join(parts) if parts else 'RAM unavailable'

def _vram_status():
    try:
        result = subprocess.run(
            [
                'nvidia-smi',
                '--query-gpu=memory.used,memory.total,utilization.gpu',
                '--format=csv,noheader,nounits',
            ],
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
            check=False,
        )
        line = result.stdout.strip().splitlines()[0]
        used, total, util = [part.strip() for part in line.split(',')[:3]]
        return f'VRAM {int(used) / 1024:.1f}/{int(total) / 1024:.1f} GB | GPU {util}%'
    except Exception:
        try:
            if torch.cuda.is_available():
                free, total = torch.cuda.mem_get_info()
                used = total - free
                return f'VRAM {_gb(used):.1f}/{_gb(total):.1f} GB'
        except Exception:
            pass
    return 'VRAM unavailable'

def _reader_thread(stream, output_queue):
    try:
        for line in stream:
            output_queue.put(line)
    finally:
        output_queue.put(None)

def _update_iteration_from_line(line, current):
    match = _ITER_RE.search(line)
    if not match:
        return current
    iteration = int(match.group(1))
    total = int(match.group(2)) if match.group(2) else current[1]
    return iteration, total

def _render_train_monitor(tail, process=None, iteration_state=(0, ITERATIONS), returncode=None):
    clear_output(wait=True)
    status = 'running' if returncode is None else f'exit code {returncode}'
    iteration, total = iteration_state
    print(f'Training monitor: {status}')
    print(f'iteration: {iteration}/{total}')
    print(_ram_status(process))
    print(_vram_status())
    print(f'\n===== train.py output tail: last {tail.maxlen} lines =====')
    if tail:
        print('\n'.join(tail))
    else:
        print('(waiting for train.py output)')

def _renderer_ws_url():
    renderer_ws_url = globals().get('RENDERER_WS_URL', '').strip()
    if not renderer_ws_url or 'YOUR_PUBLIC_HOST' in renderer_ws_url:
        raise ValueError('Set RENDERER_WS_URL in the renderer relay check cell before training, or set STREAM_TO_RENDERER = False.')
    return renderer_ws_url

def _run_monitored_process(cmd, env, cwd, total_iterations=ITERATIONS, tail_lines=TRAIN_OUTPUT_TAIL_LINES):
    process = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        text=True,
        encoding='utf-8',
        errors='replace',
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    tail = deque(maxlen=tail_lines)
    output_queue = queue.Queue()
    iteration_state = (0, total_iterations)
    assert process.stdout is not None
    reader = threading.Thread(target=_reader_thread, args=(process.stdout, output_queue), daemon=True)
    reader.start()
    last_render = 0.0
    stream_done = False
    while not stream_done or process.poll() is None or not output_queue.empty():
        try:
            line = output_queue.get(timeout=0.25)
        except queue.Empty:
            line = None
        if line is None:
            stream_done = stream_done or process.poll() is not None
        else:
            line = line.rstrip('\n')
            tail.append(line)
            iteration_state = _update_iteration_from_line(line, iteration_state)
        now = time.monotonic()
        if now - last_render >= TRAIN_STATUS_INTERVAL_SECONDS:
            _render_train_monitor(tail, process, iteration_state)
            last_render = now
    returncode = process.wait()
    reader.join(timeout=1.0)
    _render_train_monitor(tail, process, iteration_state, returncode=returncode)
    print('===== exit code:', returncode, '=====')
    if returncode != 0:
        raise RuntimeError('train.py failed. Read the output tail above for the real error.')

def _base_env():
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(GS_ROOT), env.get('PYTHONPATH', '')])
    env.setdefault('PYTHONUTF8', '1')
    env.setdefault('PYTHONIOENCODING', 'utf-8')
    return env

if FRAMEWORK_MODE == 'osn_gs':
    osn_train_script = GS_ROOT / 'train.py'
    if not osn_train_script.exists():
        raise FileNotFoundError(f'Missing OSN-GS train.py: {osn_train_script}')

    cmd = [
        sys.executable, str(osn_train_script),
        '-m', str(MODEL_ROOT),
        '--iterations', str(ITERATIONS),
        '--save_iterations', *map(str, SAVE_ITERATIONS),
        '--test_iterations', *map(str, TEST_ITERATIONS),
        '--base_curve_count', str(OSN_BASE_CURVE_COUNT),
        '--visible_surface_resolution_u', str(OSN_VISIBLE_SURFACE_RESOLUTION_U),
        '--visible_surface_resolution_v', str(OSN_VISIBLE_SURFACE_RESOLUTION_V),
        '--visible_surface_resolution_scale', str(OSN_VISIBLE_SURFACE_RESOLUTION_SCALE),
        '--visible_surface_fit_device', str(OSN_VISIBLE_SURFACE_FIT_DEVICE),
        '--visible_surface_fit_chunk_size', str(OSN_VISIBLE_SURFACE_FIT_CHUNK_SIZE),
        '--max_surface_control_points', str(OSN_MAX_SURFACE_CONTROL_POINTS),
        '--voxel_grid_resolution', str(OSN_VOXEL_GRID_RESOLUTION),
        '--voxel_max_subdivision_depth', str(OSN_VOXEL_MAX_SUBDIVISION_DEPTH),
        '--voxel_density_quantile', str(OSN_VOXEL_DENSITY_QUANTILE),
        '--voxel_density_covariance_weight_cap', str(OSN_VOXEL_DENSITY_COVARIANCE_WEIGHT_CAP),
        '--voxel_normal_knn', str(OSN_VOXEL_NORMAL_KNN),
        '--voxel_boundary_angle_degrees', str(OSN_VOXEL_BOUNDARY_ANGLE_DEGREES),
        '--voxel_min_points_per_region', str(OSN_VOXEL_MIN_POINTS_PER_REGION),
        '--covariance_init', str(OSN_COVARIANCE_INIT),
        '--covariance_knn_chunk_size', str(OSN_COVARIANCE_KNN_CHUNK_SIZE),
        '--covariance_min_scale', str(OSN_COVARIANCE_MIN_SCALE),
        '--covariance_max_scale_ratio', str(OSN_COVARIANCE_MAX_SCALE_RATIO),
        '--covariance_scale_multiplier', str(OSN_COVARIANCE_SCALE_MULTIPLIER),
        '--uncertain_samples_u', str(OSN_UNCERTAIN_SAMPLES_U),
        '--uncertain_samples_v', str(OSN_UNCERTAIN_SAMPLES_V),
        '--surface_update_interval', str(OSN_SURFACE_UPDATE_INTERVAL),
        '--surface_loss_patch_budget', str(OSN_SURFACE_LOSS_PATCH_BUDGET),
        '--surface_residual_ratio_threshold', str(OSN_SURFACE_RESIDUAL_RATIO_THRESHOLD),
        '--surface_residual_patience', str(OSN_SURFACE_RESIDUAL_PATIENCE),
        '--surface_local_min_gaussians', str(OSN_SURFACE_LOCAL_MIN_GAUSSIANS),
        '--surface_local_min_component', str(OSN_SURFACE_LOCAL_MIN_COMPONENT),
        '--surface_fit_mode', str(OSN_SURFACE_FIT_MODE),
        '--surface_degree_u', str(OSN_SURFACE_DEGREE_U),
        '--surface_degree_v', str(OSN_SURFACE_DEGREE_V),
        '--surface_fit_smoothness', str(OSN_SURFACE_FIT_SMOOTHNESS),
        '--surface_fit_tikhonov', str(OSN_SURFACE_FIT_TIKHONOV),
        '--surface_fit_rounds', str(OSN_SURFACE_FIT_ROUNDS),
        '--surface_projection_iterations', str(OSN_SURFACE_PROJECTION_ITERATIONS),
        '--density_control_interval', str(OSN_DENSITY_CONTROL_INTERVAL),
        '--progress_log_interval', str(PROGRESS_LOG_INTERVAL),
        '--timing_log_interval', str(TIMING_LOG_INTERVAL),
        '--densify_from_iter', str(OSN_DENSIFY_FROM_ITER),
        '--densify_until_iter', str(OSN_DENSIFY_UNTIL_ITER),
        '--densification_interval', str(OSN_DENSIFICATION_INTERVAL),
        '--densify_grad_threshold', str(OSN_DENSIFY_GRAD_THRESHOLD),
        '--adc_max_gaussians', str(OSN_ADC_MAX_GAUSSIANS),
        '--adc_percent_dense', str(OSN_ADC_PERCENT_DENSE),
        '--adc_prune_opacity_threshold', str(OSN_ADC_PRUNE_OPACITY_THRESHOLD),
        '--adc_split_samples', str(OSN_ADC_SPLIT_SAMPLES),
        '--adc_max_screen_size', str(OSN_ADC_MAX_SCREEN_SIZE),
        '--adc_max_scale_ratio', str(OSN_ADC_MAX_SCALE_RATIO),
        '--opacity_reset_interval', str(OSN_OPACITY_RESET_INTERVAL),
        '--screen_size_prune_from_iter', str(OSN_SCREEN_SIZE_PRUNE_FROM_ITER),
        '-s', str(DATA_ROOT),
        '--image_downscale', str(OSN_IMAGE_DOWNSCALE),
        '--max_images', str(OSN_MAX_IMAGES),
        '--device', str(OSN_DEVICE),
        '--image_device', str(OSN_IMAGE_DEVICE),
        '--train_resolution_scale', str(OSN_TRAIN_RESOLUTION_SCALE),
        '--max_uncertain_gaussians', str(OSN_MAX_UNCERTAIN_GAUSSIANS),
    ]
    if not OSN_USE_VOXEL_SURFACE_REGIONS:
        cmd += ['--disable_voxel_surface_regions']
    if not OSN_ADAPTIVE_VOXEL_DENSITY:
        cmd += ['--disable_adaptive_voxel_density']
    if not OSN_ENABLE_LOCAL_SURFACE_CORRECTION:
        cmd += ['--disable_local_surface_correction']
    if OSN_RESUME_CHECKPOINT:
        cmd += ['--resume_checkpoint', str(OSN_RESUME_CHECKPOINT)]
    if OSN_LOW_VRAM:
        cmd += ['--low_vram']
    if OSN_DISABLE_CUDA_RASTERIZER:
        cmd += ['--disable_cuda_rasterizer']
    cmd += ['--stream_every', str(STREAM_EVERY)]
    cmd += ['--stream_iterations', *map(str, STREAM_ITERATIONS)]
    cmd += ['--stream_max_gaussians', str(STREAM_MAX_GAUSSIANS)]
    cmd += ['--stream_cache_dir', str(STREAM_CACHE_DIR)]
    if STREAM_TO_RENDERER:
        cmd += ['--stream_url', _renderer_ws_url()]
    if not STREAM_NURBS:
        cmd += ['--disable_stream_nurbs']
    if not WRITE_OUTPUT_FILES:
        cmd += ['--disable_output_files']

    env = _base_env()
    if os.name == 'nt':
        cuda_home = os.environ.get('CUDA_HOME', '').strip()
        if cuda_home and Path(cuda_home).exists():
            env['CUDA_HOME'] = cuda_home
            env['PATH'] = os.pathsep.join([str(Path(cuda_home) / 'bin'), env['PATH']])
    env.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    env.setdefault('MAX_JOBS', '2')
    try:
        if torch.cuda.is_available():
            major, minor = torch.cuda.get_device_capability(0)
            env.setdefault('TORCH_CUDA_ARCH_LIST', f'{major}.{minor}')
    except Exception as exc:
        print('Warning: could not detect CUDA arch for OSN-GS rasterizer build:', exc)

    print('Running OSN-GS:', ' '.join(cmd))
    print('GS_ROOT:', GS_ROOT)
    print('MODEL_ROOT:', MODEL_ROOT)
    print('COLMAP DATA_ROOT:', DATA_ROOT)
    print('TRAIN DEVICE:', OSN_DEVICE)
    print('IMAGE DEVICE:', OSN_IMAGE_DEVICE)
    print('NURBS FIT DEVICE:', OSN_VISIBLE_SURFACE_FIT_DEVICE)
    print('VOXEL SURFACE REGIONS:', OSN_USE_VOXEL_SURFACE_REGIONS)
    print('VOXEL GRID:', OSN_VOXEL_GRID_RESOLUTION, 'BOUNDARY ANGLE:', OSN_VOXEL_BOUNDARY_ANGLE_DEGREES)
    print('COVARIANCE INIT:', OSN_COVARIANCE_INIT)
    print('COVARIANCE KNN CHUNK:', OSN_COVARIANCE_KNN_CHUNK_SIZE)
    print('COVARIANCE SCALE MULTIPLIER:', OSN_COVARIANCE_SCALE_MULTIPLIER)
    print('Streaming:', 'enabled' if STREAM_TO_RENDERER else 'disabled')
    if STREAM_TO_RENDERER:
        print('Renderer WS URL:', _renderer_ws_url())
        print('Stream every:', STREAM_EVERY)
        print('Stream iterations:', sorted(STREAM_ITERATIONS))
        print('Stream max gaussians:', STREAM_MAX_GAUSSIANS or 'unlimited')
        print('Stream NURBS:', STREAM_NURBS)
    print('Stream cache dir:', STREAM_CACHE_DIR)
    print('Output files:', 'enabled' if WRITE_OUTPUT_FILES else 'disabled')
    _run_monitored_process(cmd, env, GS_ROOT, total_iterations=ITERATIONS)

else:
    def patch_lazy_colmap_image_loading(project_root):
        dataset_readers = project_root / 'scene' / 'dataset_readers.py'
        camera_utils = project_root / 'utils' / 'camera_utils.py'
        if not PATCH_LAZY_COLMAP_IMAGES or not dataset_readers.exists() or not camera_utils.exists():
            return
        readers_text = dataset_readers.read_text()
        old_read = '''        image = Image.open(image_path)\n        # get rid of too many opened files\n        image = copy.deepcopy(image)\n        cam_info = CameraInfo(uid=uid, R=R, T=T, FovY=FovY, FovX=FovX, image=image,\n                              image_path=image_path, image_name=image_name, width=width, height=height)\n'''
        new_read = '''        # RAM patch: keep only the path here. loadCam() opens the image lazily.\n        image = None\n        cam_info = CameraInfo(uid=uid, R=R, T=T, FovY=FovY, FovX=FovX, image=image,\n                              image_path=image_path, image_name=image_name, width=width, height=height)\n'''
        if old_read in readers_text:
            dataset_readers.write_text(readers_text.replace(old_read, new_read, 1))
            print('Patched COLMAP camera reader for lazy image loading:', dataset_readers)
        camera_text = camera_utils.read_text()
        if 'from PIL import Image\n' not in camera_text:
            camera_text = camera_text.replace('import numpy as np\n', 'import numpy as np\nfrom PIL import Image\n', 1)
        old_size = '''def loadCam(args, id, cam_info, resolution_scale):\n    orig_w, orig_h = cam_info.image.size\n'''
        new_size = '''def loadCam(args, id, cam_info, resolution_scale):\n    image = cam_info.image\n    if image is None:\n        image = Image.open(cam_info.image_path)\n    orig_w, orig_h = image.size\n'''
        if old_size in camera_text:
            camera_text = camera_text.replace(old_size, new_size, 1)
        camera_text = camera_text.replace('    resized_image_rgb = PILtoTorch(cam_info.image, resolution)\n', '    resized_image_rgb = PILtoTorch(image, resolution)\n', 1)
        camera_utils.write_text(camera_text)
        print('Patched camera loader for lazy image open:', camera_utils)

    patch_lazy_colmap_image_loading(GS_ROOT)
    train_runner = Path('/content/train_with_ws_runtime.py') if IS_COLAB else NOTEBOOK_ROOT / 'train_with_ws_runtime.py'
    train_runner.write_text(r'''
import atexit
import importlib
import json
import os
import runpy
import sys
import time
import torch

SH_C0 = 0.28209479177387814
WS_URL = os.environ.get('GS_STREAM_WS_URL', '').strip()
STREAM_EVERY = int(os.environ.get('GS_STREAM_EVERY', '0') or '0')
STREAM_ITERATIONS = {int(value) for value in os.environ.get('GS_STREAM_ITERATIONS', '').split(',') if value.strip()}
STREAM_MAX_GAUSSIANS = int(os.environ.get('GS_STREAM_MAX_GAUSSIANS', '0') or '0')
STREAM_CACHE_DIR = os.environ.get('GS_STREAM_CACHE_DIR', '').strip()
TRAIN_SCRIPT = os.environ.get('GS_TRAIN_SCRIPT', 'train.py')
GAUSSIAN_MODEL_IMPORT = os.environ.get('GS_GAUSSIAN_MODEL_IMPORT', 'scene.gaussian_model:GaussianModel')
GAUSSIAN_STREAM_HOOK = os.environ.get('GS_GAUSSIAN_STREAM_HOOK', 'update_learning_rate')
_ws = None
_last_error_at = 0.0

def import_symbol(spec):
    module_name, _, attr_name = spec.partition(':')
    module = importlib.import_module(module_name)
    return getattr(module, attr_name)

GaussianModel = import_symbol(GAUSSIAN_MODEL_IMPORT)

def should_stream(iteration):
    if not WS_URL and not STREAM_CACHE_DIR:
        return False
    if iteration in STREAM_ITERATIONS:
        return True
    return STREAM_EVERY > 0 and iteration % STREAM_EVERY == 0

def get_socket():
    global _ws
    if _ws is not None:
        return _ws
    from websockets.sync.client import connect
    _ws = connect(WS_URL, max_size=None, open_timeout=10, close_timeout=2)
    try:
        _ws.recv(timeout=1)
    except Exception:
        pass
    print(f'[WS] connected to renderer relay: {WS_URL}', flush=True)
    return _ws

def close_socket():
    global _ws
    if _ws is not None:
        try:
            _ws.close()
        except Exception:
            pass
        _ws = None

atexit.register(close_socket)

def selected_indices(count, device):
    if STREAM_MAX_GAUSSIANS > 0 and count > STREAM_MAX_GAUSSIANS:
        return torch.linspace(0, count - 1, steps=STREAM_MAX_GAUSSIANS, device=device).long()
    return slice(None)

def snapshot_payload(gaussians, iteration):
    with torch.no_grad():
        xyz_all = gaussians.get_xyz
        total_count = int(xyz_all.shape[0])
        idx = selected_indices(total_count, xyz_all.device)
        xyz = xyz_all[idx].detach().float().cpu()
        scaling = gaussians.get_scaling[idx].detach().float().cpu()
        rotation = gaussians.get_rotation[idx].detach().float().cpu()
        opacity = gaussians.get_opacity[idx].detach().float().reshape(-1).cpu()
        fdc = gaussians.get_features_dc[idx, 0, :].detach().float().cpu()
        color = torch.clamp(0.5 + SH_C0 * fdc, 0.0, 1.0)
    count = int(xyz.shape[0])
    return {
        'type': 'snapshot',
        'iteration': int(iteration),
        'parameterSpace': 'render',
        'count': count,
        'positions': xyz.reshape(-1).tolist(),
        'scales': scaling.reshape(-1).tolist(),
        'colors': color.reshape(-1).tolist(),
        'opacities': opacity.reshape(-1).tolist(),
        'rotations': rotation.reshape(-1).tolist(),
        'metadata': {
            'source': 'colab-training',
            'totalCount': total_count,
            'sentCount': count,
            'capped': count != total_count,
        },
    }

def cache_snapshot(payload):
    if not STREAM_CACHE_DIR:
        return
    from pathlib import Path
    cache_dir = Path(STREAM_CACHE_DIR)
    cache_dir.mkdir(parents=True, exist_ok=True)
    iteration = int(payload.get('iteration', 0))
    (cache_dir / f'{iteration:08d}.json').write_text(json.dumps(payload, ensure_ascii=False, separators=(',', ':')), encoding='utf-8')

def stream_snapshot(gaussians, iteration):
    global _last_error_at
    if not should_stream(iteration):
        return
    try:
        payload = snapshot_payload(gaussians, iteration)
        cache_snapshot(payload)
        if not WS_URL:
            return
        get_socket().send(json.dumps(payload, separators=(',', ':')))
        capped = ' capped' if payload['metadata']['capped'] else ''
        print(f"[WS] sent iteration {iteration}: {payload['count']}/{payload['metadata']['totalCount']} gaussians{capped}", flush=True)
    except Exception as exc:
        now = time.time()
        if now - _last_error_at > 10:
            print(f'[WS] stream/cache failed at iteration {iteration}: {exc}', flush=True)
            _last_error_at = now
        close_socket()

_original_stream_hook = getattr(GaussianModel, GAUSSIAN_STREAM_HOOK)

def stream_hook_wrapper(self, iteration, *args, **kwargs):
    result = _original_stream_hook(self, iteration, *args, **kwargs)
    stream_snapshot(self, iteration)
    return result

setattr(GaussianModel, GAUSSIAN_STREAM_HOOK, stream_hook_wrapper)
sys.argv[0] = TRAIN_SCRIPT
runpy.run_path(TRAIN_SCRIPT, run_name='__main__')
''', encoding='utf-8')

    cmd = [
        sys.executable, str(train_runner),
        '-s', str(DATA_ROOT),
        '-m', str(MODEL_ROOT),
        '--iterations', str(ITERATIONS),
        '--save_iterations', *map(str, SAVE_ITERATIONS),
        '--test_iterations', *map(str, TEST_ITERATIONS),
        '--densify_until_iter', str(DENSIFY_UNTIL_ITER),
        '--densification_interval', str(DENSIFICATION_INTERVAL),
        '--densify_grad_threshold', str(DENSIFY_GRAD_THRESHOLD),
        *map(str, TRAIN_EXTRA_ARGS),
    ]

    def resolve_project_path(value):
        path = Path(value)
        return path if path.is_absolute() else GS_ROOT / path

    extensions = [normalize_extension(spec) for spec in CUDA_EXTENSIONS if normalize_extension(spec).get('enabled', True)]
    extension_paths = [GS_ROOT]
    extension_paths += [resolve_project_path(spec['path']) for spec in extensions]
    extension_paths += [resolve_project_path(path) for path in EXTRA_PYTHONPATHS]

    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join(map(str, extension_paths + [Path(p) for p in env.get('PYTHONPATH', '').split(os.pathsep) if p]))
    env['GS_TRAIN_SCRIPT'] = str(TRAIN_SCRIPT)
    env['GS_GAUSSIAN_MODEL_IMPORT'] = str(GAUSSIAN_MODEL_IMPORT)
    env['GS_GAUSSIAN_STREAM_HOOK'] = str(GAUSSIAN_STREAM_HOOK)
    env['GS_STREAM_EVERY'] = str(STREAM_EVERY)
    env['GS_STREAM_ITERATIONS'] = ','.join(map(str, STREAM_ITERATIONS))
    env['GS_STREAM_MAX_GAUSSIANS'] = str(STREAM_MAX_GAUSSIANS)
    env['GS_STREAM_CACHE_DIR'] = str(STREAM_CACHE_DIR)
    if STREAM_TO_RENDERER:
        env['GS_STREAM_WS_URL'] = _renderer_ws_url()
    else:
        env.pop('GS_STREAM_WS_URL', None)

    print('Running:', ' '.join(cmd))
    print('GS_ROOT:', GS_ROOT)
    print('DATA_ROOT:', DATA_ROOT)
    print('MODEL_ROOT:', MODEL_ROOT)
    print('TRAIN_SCRIPT:', TRAIN_SCRIPT)
    print('GAUSSIAN_MODEL_IMPORT:', GAUSSIAN_MODEL_IMPORT)
    print('GAUSSIAN_STREAM_HOOK:', GAUSSIAN_STREAM_HOOK)
    print('Streaming:', 'enabled' if STREAM_TO_RENDERER else 'disabled')
    if STREAM_TO_RENDERER:
        print('Renderer WS URL:', env['GS_STREAM_WS_URL'])
        print('Stream every:', STREAM_EVERY)
        print('Stream iterations:', sorted(STREAM_ITERATIONS))
        print('Stream max gaussians:', STREAM_MAX_GAUSSIANS or 'unlimited')
    print('PYTHONPATH extension paths:')
    for path in extension_paths:
        print(' ', path)
    _run_monitored_process(cmd, env, GS_ROOT, total_iterations=ITERATIONS, tail_lines=200)

    point_cloud_root = MODEL_ROOT / 'point_cloud'
    renamed = []
    if point_cloud_root.exists():
        for path in sorted(point_cloud_root.glob('iteration_*')):
            if not path.is_dir():
                continue
            iteration = path.name.removeprefix('iteration_')
            if not iteration.isdigit():
                continue
            target = point_cloud_root / iteration
            if target.exists():
                continue
            path.rename(target)
            renamed.append((path.name, target.name))
    if renamed:
        print('Renamed point cloud folders:')
        for old, new in renamed:
            print(f'  {old} -> {new}')

Training monitor: exit code 0
iteration: 10000/10000
RAM 26.9/63.6 GB (42%)
VRAM 5.0/15.9 GB | GPU 15%

===== train.py output tail: last 40 lines =====
OSN-GS surface maintenance: iteration=9000 checked=107 max_residual=0.0562492 candidates=1 corrected=1 patches=116 uv_refreshed=217674
OSN-GS ADC: iteration=9000 cloned=2714 split=28 pruned=407 opacity=407 screen=0 world=0 gaussians=219995 tracked=208267 max_grad=0.00379493 mean_grad=1.55796e-05 threshold=0.0002
OSN-GS ADC: iteration=9000 opacity_reset=0.01
OSN-GS progress: iteration=9000/10000 loss=0.438369 psnr=19.346 gaussians=219995 uncertain=0
OSN-GS timing: iteration=9000 sample=0.049s render_loss=0.012s surface_loss=0.062s backward=0.086s optim=0.002s density=4.202s save=0.024s log=0.001s total=4.437s avg_iter=0.187s
[WS] sent iteration 9000: 219995/219995 gaussians + nurbs
OSN-GS ADC: iteration=9100 cloned=422 split=42 pruned=42921 opacity=42921 screen=0 world=0 gaussians=177517 tracked=216698 max_grad=0.00313867 mean_grad=6.251

In [28]:
# Manual bulk streaming of cached Gaussian snapshots.
# Run this after the training cell to resend all cached iterations at once.
#
# Live training paces snapshots naturally: real wall-clock time passes
# between training iterations, so the relay/ngrok tunnel/browser always have
# time to fully receive, JSON.parse, and re-render a large (tens of MB)
# snapshot before the next one arrives. This cell replays the exact same
# cached payloads back-to-back instead, so it needs its own pacing or it can
# out-run the relay/browser and the browser never finishes rendering a
# snapshot before the next one lands. Capping STREAM_MAX_GAUSSIANS is NOT the
# fix here -- it silently degrades scene quality. Pacing to match how much
# data the pipe can actually absorb per second is.
import json
import time
from pathlib import Path

BULK_STREAM_CACHE_DIR = Path(globals().get('STREAM_CACHE_DIR', MODEL_ROOT / 'stream_cache'))
BULK_STREAM_ITERATIONS = []  # Empty means every cached iteration. Example: [1000, 10000, 20000]
BULK_STREAM_MIN_DELAY_SECONDS = 1.0  # Floor delay between sends, regardless of payload size.
BULK_STREAM_TARGET_BYTES_PER_SECOND = 4 * 1024 * 1024  # Conservative pacing budget for relay+browser.
BULK_STREAM_WS_URL = globals().get('RENDERER_WS_URL', '')

if not BULK_STREAM_WS_URL:
    raise ValueError('Set RENDERER_WS_URL in the renderer relay check cell before bulk streaming.')
if not BULK_STREAM_CACHE_DIR.exists():
    raise FileNotFoundError(f'Missing stream cache dir: {BULK_STREAM_CACHE_DIR}')

wanted = {int(v) for v in BULK_STREAM_ITERATIONS}
files = []
for path in sorted(BULK_STREAM_CACHE_DIR.glob('*.json')):
    try:
        iteration = int(path.stem)
    except ValueError:
        continue
    if wanted and iteration not in wanted:
        continue
    files.append((iteration, path))

if not files:
    raise RuntimeError(f'No cached stream snapshots found in {BULK_STREAM_CACHE_DIR}')

from websockets.sync.client import connect

print('Bulk stream cache:', BULK_STREAM_CACHE_DIR)
print('Bulk stream iterations:', [iteration for iteration, _ in files])
print('Bulk stream target:', BULK_STREAM_WS_URL)

with connect(BULK_STREAM_WS_URL, max_size=None, open_timeout=10, close_timeout=2) as ws:
    try:
        ws.recv(timeout=1)
    except Exception:
        pass
    for index, (iteration, path) in enumerate(files, start=1):
        raw_text = path.read_text(encoding='utf-8')
        payload = json.loads(raw_text)
        message = json.dumps(payload, separators=(',', ':'))
        ws.send(message)
        count = payload.get('count', '?')
        total = payload.get('metadata', {}).get('totalCount', '?')
        size_mb = len(message) / (1024 * 1024)
        delay = max(
            BULK_STREAM_MIN_DELAY_SECONDS,
            len(message) / BULK_STREAM_TARGET_BYTES_PER_SECOND,
        )
        print(
            f'[{index}/{len(files)}] sent iteration {iteration}: '
            f'{count}/{total} gaussians, {size_mb:.1f} MB, next in {delay:.2f}s'
        )
        if index < len(files):
            time.sleep(delay)

Bulk stream cache: C:\Projects\OSN-GS\output\osn_gs_scene\stream_cache
Bulk stream iterations: [200, 400, 600, 800, 1000, 1200, 1400, 1600, 1800, 2000, 2200, 2400, 2600, 2800, 3000, 3200, 3400, 3600, 3800, 4000, 4200, 4400, 4600, 4800, 5000, 5200, 5400, 5600, 5800, 6000, 6200, 6400, 6600, 6800, 7000, 7200, 7400, 7600, 7800, 8000, 8200, 8400, 8600, 8800, 9000, 9200, 9400, 9600, 9800, 10000]
Bulk stream target: wss://cotton-credible-primer.ngrok-free.dev/garden


InvalidStatus: server rejected WebSocket connection: HTTP 403

## 6. Inspect and Download Outputs

In [ ]:
from pathlib import Path

def _osn_output_sort_key(path):
    name = path.parent.name
    if name == 'final':
        return (2, 0)
    if name.isdigit():
        return (1, int(name))
    if name.startswith('iteration_') and name.removeprefix('iteration_').isdigit():
        return (1, int(name.removeprefix('iteration_')))
    return (0, name)

if FRAMEWORK_MODE == 'osn_gs':
    ply_files = sorted(MODEL_ROOT.glob('**/point_cloud.ply'), key=_osn_output_sort_key)
else:
    ply_files = sorted(MODEL_ROOT.glob('point_cloud/[0-9]*/point_cloud.ply'), key=lambda path: int(path.parent.name))

print('PLY files:')
for path in ply_files:
    print(path)

if ply_files:
    latest = ply_files[-1]
    print('\nLatest PLY header:')
    with latest.open('rb') as f:
        for _ in range(40):
            line = f.readline().decode('utf-8', errors='replace').rstrip()
            print(line)
            if line == 'end_header':
                break

if FRAMEWORK_MODE == 'osn_gs':
    metrics_files = sorted(MODEL_ROOT.glob('**/metrics.txt'), key=_osn_output_sort_key)
    nurbs_files = sorted(MODEL_ROOT.glob('**/nurbs_surface.json'), key=_osn_output_sort_key)
    print('\nMetrics files:')
    for path in metrics_files:
        print(path)
    if metrics_files:
        print('\nLatest metrics:')
        print(metrics_files[-1].read_text())

    print('\nNURBS intermediate files:')
    for path in nurbs_files:
        print(path)
    if nurbs_files:
        import json
        latest_nurbs = json.loads(nurbs_files[-1].read_text(encoding='utf-8'))
        print('\nLatest NURBS intermediate:')
        print('type:', latest_nurbs.get('type'))
        print('iteration:', latest_nurbs.get('iteration'))
        print('control_grid_shape:', latest_nurbs.get('control_grid_shape'))
        print('degree:', latest_nurbs.get('degree_u'), latest_nurbs.get('degree_v'))

if FRAMEWORK_MODE == 'osn_gs':
    import json

    output_pairs = []
    for ply_path in ply_files:
        run_dir = ply_path.parent
        nurbs_path = run_dir / 'nurbs_surface.json'
        metrics_path = run_dir / 'metrics.txt'
        checkpoint_path = run_dir / 'checkpoint.pt'
        output_pairs.append({
            'name': run_dir.name,
            'gaussian_ply': str(ply_path.relative_to(MODEL_ROOT)),
            'nurbs_surface': str(nurbs_path.relative_to(MODEL_ROOT)) if nurbs_path.exists() else None,
            'metrics': str(metrics_path.relative_to(MODEL_ROOT)) if metrics_path.exists() else None,
            'checkpoint': str(checkpoint_path.relative_to(MODEL_ROOT)) if checkpoint_path.exists() else None,
        })

    manifest = {
        'type': 'osn_gs_visualization_manifest',
        'model_root': str(MODEL_ROOT),
        'latest': output_pairs[-1] if output_pairs else None,
        'outputs': output_pairs,
        'notes': {
            'gaussian_ply': 'Main Gaussian primitive output.',
            'nurbs_surface': 'Visible NURBS-like intermediate saved for visualization; final output remains Gaussian.',
        },
    }
    manifest_path = MODEL_ROOT / 'visualization_manifest.json'
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    print('\nVisualization manifest:')
    print(manifest_path)
    if manifest['latest']:
        print('Latest Gaussian PLY:', manifest['latest']['gaussian_ply'])
        print('Latest NURBS surface:', manifest['latest']['nurbs_surface'])



In [ ]:
import shutil
from pathlib import Path

if not MODEL_ROOT.exists():
    raise FileNotFoundError(f'MODEL_ROOT does not exist yet: {MODEL_ROOT}')

project_root = Path(GS_ROOT if GS_ROOT is not None else NOTEBOOK_ROOT).resolve()
archive_dir = Path('/content') if IS_COLAB else project_root
archive_dir.mkdir(parents=True, exist_ok=True)
archive_base = archive_dir / f'{MODEL_ROOT.name}_3dgs_output'
manifest_path = MODEL_ROOT / 'visualization_manifest.json'
if FRAMEWORK_MODE == 'osn_gs' and not manifest_path.exists():
    raise FileNotFoundError('Run the output inspection cell first so visualization_manifest.json is created.')

archive_path = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=MODEL_ROOT)).resolve()
print('Created:', archive_path)
if FRAMEWORK_MODE == 'osn_gs':
    print('Included visualization_manifest.json:', manifest_path.exists())
    print('Included NURBS files:', len(list(MODEL_ROOT.glob('**/nurbs_surface.json'))))

if IS_COLAB:
    from google.colab import files
    files.download(str(archive_path))
else:
    print('Local notebook run detected; output zip was written to the project root:')
    print(archive_path)
    try:
        from IPython.display import FileLink, display
        display(FileLink(str(archive_path)))
    except Exception:
        pass
